# Anggota 3 — Retrieval

Affan Abyarahman · 245150207111020

# 1. Data Retrieval

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

with open("../Anggota1_Korpus/output/passages.jsonl", encoding="utf-8") as file:
    passages = [json.loads(line) for line in file]
with open("../Anggota1_Korpus/output/dokumen.json", encoding="utf-8") as file:
    dokumen = json.load(file)
with open("../Anggota2_Pertanyaan/output/passages_preprocessed.jsonl", encoding="utf-8") as file:
    passage_bertoken = [json.loads(line) for line in file]
with open("../Anggota2_Pertanyaan/output/hasil_pertanyaan.json", encoding="utf-8") as file:
    hasil_pertanyaan = json.load(file)

Indeks memakai token dari Anggota 2. ID passage menghubungkan token dengan teks dan metadata Anggota 1.

# 2. Ranking TF-IDF

In [2]:
token_by_id = {p["passage_id"]: p["token_pencarian"] for p in passage_bertoken}
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
matriks = vectorizer.fit_transform([token_by_id[p["id"]] for p in passages])

baris_pencarian = []
for soal in hasil_pertanyaan:
    query = soal["query"]
    urutan = []
    kandidat_top5 = []
    if query["status"] == "SIAP_RETRIEVAL" and query["token_pencarian"]:
        vektor = vectorizer.transform([query["token_pencarian"]])
        skor = cosine_similarity(vektor, matriks)[0]
        urutan = sorted(
            (i for i, nilai in enumerate(skor) if nilai > 0),
            key=lambda i: (-skor[i], passages[i]["id"]),
        )
        kandidat_top5 = [
            {
                **passages[i],
                "passage_id": passages[i]["id"],
                "rank": rank,
                "score": float(skor[i]),
                "sumber": dokumen[passages[i]["dok_id"]],
            }
            for rank, i in enumerate(urutan[:5], start=1)
        ]
    baris_pencarian.append({
        **soal,
        "urutan_passage": [passages[i]["id"] for i in urutan],
        "kandidat_top5": kandidat_top5,
    })

hasil_pencarian = {"jumlah_passage": len(passages), "hasil_per_pertanyaan": baris_pencarian}

pd.DataFrame(
    {"id_pertanyaan": soal["id_pertanyaan"], "rank": p["rank"], "passage_id": p["passage_id"], "score": p["score"]}
    for soal in baris_pencarian for p in soal["kandidat_top5"]
)

,id_pertanyaan,rank,passage_id,score
0,A2-001,1,pa2020-III-J-2.1,0.264577
1,A2-001,2,pa2020-IV-A-3,0.244562
2,A2-001,3,pa2020-II-I-7,0.241728
3,A2-001,4,pa2020-VI-D-4,0.235918
4,A2-001,5,pa2020-II-F-4,0.233342
...,...,...,...,...
95,SUP-005,1,pa2020-tabel1-1,0.681192
96,SUP-005,2,pa2020-tabel1-2,0.667080
97,SUP-005,3,pa2020-tabel1-3,0.577617
98,SUP-005,4,pa2020-tabel1-4,0.576349


Tabel menunjukkan lima kandidat dan skor cosine setiap soal. Urutan semua kandidat berskor positif dipakai untuk menghitung peringkat bukti pertama.

# 3. Simpan Kandidat

In [3]:
Path("output").mkdir(exist_ok=True)
with open("output/hasil_pencarian.json", "w", encoding="utf-8") as file:
    json.dump(hasil_pencarian, file, ensure_ascii=False, indent=2)
    file.write("\n")

Lima kandidat lengkap diteruskan ke ekstraksi. Urutan ID seluruh kandidat diteruskan ke evaluasi retrieval.